# Task 02 — KPI Dictionary & Data Quality Profile

**Project:** Retail Orders Data Quality Review  
**Purpose:** establish measurable retail KPIs and test the supplied dataset against its data dictionary before downstream reporting.

> The supplied sample contains only 12 rows. This notebook profiles the supplied sample; it does not claim to satisfy Task 03's separate 10,000+ row requirement.

## 1. Load inputs

Place `retail-orders-raw.csv` and `retail-data-dictionary.csv` in the same folder as this notebook.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path('retail-orders-raw.csv')
DICT_PATH = Path('retail-data-dictionary.csv')
raw = pd.read_csv(DATA_PATH)
data_dictionary = pd.read_csv(DICT_PATH)
print('Raw shape:', raw.shape)
print('Dictionary shape:', data_dictionary.shape)
display(raw.head())
display(data_dictionary)

## 2. Profile completeness, duplicates, types and category values

In [ ]:
profile = pd.DataFrame({
    'column': raw.columns,
    'dtype_as_loaded': [str(raw[c].dtype) for c in raw.columns],
    'row_count': len(raw),
    'missing_count': [raw[c].isna().sum() for c in raw.columns],
    'missing_pct': [round(raw[c].isna().mean() * 100, 2) for c in raw.columns],
    'distinct_non_null': [raw[c].nunique(dropna=True) for c in raw.columns],
})
print('Exact duplicate rows:', int(raw.duplicated().sum()))
print('Duplicate order_id rows:', int(raw['order_id'].duplicated(keep=False).sum()))
display(profile)
profile.to_csv('task02_column_profile.csv', index=False)

## 3. Normalize and validate fields

This section creates a validation copy and records issues rather than silently deleting or inventing values. Missing discount is flagged for business review because the supplied dictionary says missing means zero only after justification.

In [ ]:
check = raw.copy()
issues = []
def add_issue(rule, mask, detail):
    mask = pd.Series(mask, index=check.index).fillna(False)
    for idx in check.index[mask]:
        issues.append({'source_row_index': int(idx), 'order_id': check.loc[idx, 'order_id'] if 'order_id' in check else None,
                       'rule': rule, 'observed_value': str(check.loc[idx].to_dict()), 'detail': detail})

# Normalize strings
for col in ['customer_segment', 'city', 'category', 'payment_status']:
    check[col] = check[col].astype('string').str.strip()
check['customer_segment'] = check['customer_segment'].str.title()
check['payment_status'] = check['payment_status'].str.title()

# Required fields and uniqueness
add_issue('order_id_required', check['order_id'].isna() | check['order_id'].astype('string').str.strip().eq(''), 'Order ID is required')
add_issue('duplicate_order_id', check['order_id'].duplicated(keep=False), 'Duplicate order ID; review source records before selecting a canonical row')
for col in ['order_date','customer_segment','city','category','quantity','unit_price','payment_status']:
    add_issue(f'{col}_missing', check[col].isna() | check[col].astype('string').str.strip().eq('<NA>'), f'{col} is required')

# Dates: accept common day-first or ISO strings; invalid date is not coerced into a guessed date
parsed_dates = pd.to_datetime(check['order_date'], errors='coerce', dayfirst=True)
add_issue('order_date_unparseable', parsed_dates.isna(), 'Date cannot be parsed; verify source value')
check['order_date_parsed'] = parsed_dates
add_issue('order_date_before_2025', parsed_dates.notna() & (parsed_dates < pd.Timestamp('2025-01-01')), 'Date is earlier than dictionary lower bound')

# Allowed categories after normalization
allowed_segments = {'Student','Fresher','Professional'}
allowed_categories = {'Learning Kit','Course Access','Mentor Session'}
allowed_statuses = {'Paid','Pending','Failed','Refunded'}
add_issue('customer_segment_invalid', ~check['customer_segment'].isin(allowed_segments), 'Segment is outside the approved set')
add_issue('category_invalid', ~check['category'].isin(allowed_categories), 'Category is outside the approved set')
add_issue('payment_status_invalid', ~check['payment_status'].isin(allowed_statuses), 'Payment status is outside the approved set')

# Numeric fields
check['quantity_numeric'] = pd.to_numeric(check['quantity'], errors='coerce')
add_issue('quantity_non_numeric', check['quantity_numeric'].isna(), 'Quantity must be numeric')
add_issue('quantity_not_positive_integer', check['quantity_numeric'].notna() & ((check['quantity_numeric'] <= 0) | (check['quantity_numeric'] % 1 != 0)), 'Quantity must be a positive whole number')
check['unit_price_numeric'] = pd.to_numeric(check['unit_price'], errors='coerce')
add_issue('unit_price_invalid', check['unit_price_numeric'].isna() | (check['unit_price_numeric'] < 0), 'Unit price must be numeric and non-negative')
check['discount_numeric'] = pd.to_numeric(check['discount_pct'], errors='coerce')
add_issue('discount_missing_review', check['discount_numeric'].isna(), 'Missing discount requires business justification before imputation')
add_issue('discount_out_of_range', check['discount_numeric'].notna() & ~check['discount_numeric'].between(0,100), 'Discount must be between 0 and 100')

issues_df = pd.DataFrame(issues)
print('Issue count (rule-row pairs):', len(issues_df))
display(issues_df)
issues_df.to_csv('task02_issue_log.csv', index=False)

## 4. KPI baseline (only on records passing core validity checks)

This baseline is intentionally conservative. Duplicate order IDs and records with invalid date, quantity, price, discount, category, segment or payment status are excluded from the KPI calculation and remain in the issue log. A production workflow should resolve duplicates with the source owner rather than assume one copy is correct.

In [ ]:
valid = check.copy()
valid = valid[~valid['order_id'].duplicated(keep=False)]
valid = valid[valid['order_date_parsed'].notna()]
valid = valid[valid['quantity_numeric'].notna() & (valid['quantity_numeric'] > 0) & (valid['quantity_numeric'] % 1 == 0)]
valid = valid[valid['unit_price_numeric'].notna() & (valid['unit_price_numeric'] >= 0)]
valid = valid[valid['discount_numeric'].between(0,100)]
valid = valid[valid['category'].isin(allowed_categories) & valid['customer_segment'].isin(allowed_segments) & valid['payment_status'].isin(allowed_statuses)]
valid['gross_order_value'] = valid['quantity_numeric'] * valid['unit_price_numeric']
valid['net_order_value'] = valid['gross_order_value'] * (1 - valid['discount_numeric'] / 100)

kpis = pd.DataFrame([
    ['Gross sales (pre-discount)', valid['gross_order_value'].sum()],
    ['Net order value (post-discount)', valid['net_order_value'].sum()],
    ['Distinct valid orders', valid['order_id'].nunique()],
    ['Units sold', valid['quantity_numeric'].sum()],
    ['Average order value (net)', valid['net_order_value'].sum() / valid['order_id'].nunique() if valid['order_id'].nunique() else np.nan],
    ['Average discount (%)', valid['discount_numeric'].mean()],
    ['Paid order value (net)', valid.loc[valid['payment_status'].eq('Paid'), 'net_order_value'].sum()],
    ['Refund rate (%)', 100 * valid.loc[valid['payment_status'].eq('Refunded'), 'order_id'].nunique() / valid['order_id'].nunique() if valid['order_id'].nunique() else np.nan],
    ['Payment failure rate (%)', 100 * valid.loc[valid['payment_status'].eq('Failed'), 'order_id'].nunique() / valid['order_id'].nunique() if valid['order_id'].nunique() else np.nan],
], columns=['KPI','Baseline value'])
display(kpis)
kpis.to_csv('task02_kpi_baseline.csv', index=False)
print(f'Rows in source: {len(raw)} | Rows used in conservative KPI baseline: {len(valid)}')

## 5. Data-quality contract and release decision

**Release gate:** do not publish the downstream dashboard until critical fields pass validation, duplicate IDs are resolved, and missing discount handling is approved. The attached workbook contains the KPI dictionary and the fuller field-by-field contract.

For this 12-row sample, the profile is useful for demonstrating checks but is too small to establish stable business trends. Findings should be described as sample-level observations only.

In [ ]:
print('Quality summary')
print('- Source rows:', len(raw))
print('- Exact duplicate rows:', int(raw.duplicated().sum()))
print('- Duplicate order IDs:', int(raw['order_id'].duplicated(keep=False).sum()))
print('- Missing cells:', int(raw.isna().sum().sum()))
print('- Issue log written: task02_issue_log.csv')
print('- Column profile written: task02_column_profile.csv')
print('- KPI baseline written: task02_kpi_baseline.csv')

## 6. Limitations and next actions

1. The supplied file has only 12 rows, so results are a small-sample demonstration, not a production-quality profile.
2. A duplicate `order_id` needs source-owner review; exact duplicate rows can be flagged but deletion should be documented.
3. The invalid date `2026-13-10`, negative quantity, text quantity `two`, discount above 100, missing city/date and missing discount require remediation or explicit business approval.
4. Missing discount is not automatically changed to zero because the data dictionary requires justification.
5. KPI definitions should be approved by the decision owner, especially whether refunded, pending and failed orders are included in business reporting.